# P01 · Building table (`buildings.parquet`) + ADR 0001 round 2

Builds one row per building from the BRIGHT instance labels with the package's tested loader
(`damagesar_bench.data.buildings`): pixel footprint, ground area in metres (UTM, degrees and Web Mercator handled),
the no-data rule, the JSON damage class and, for unlocked events, the label-raster majority as an agreement check.

**Before the freeze, confirmatory events get geometry only**: their damage class is never stored.

Outputs: `01_data/derived/buildings.parquet` (+ summary) and this run's folder. The summary's token columns
(`pct_lt_k_tokens_native` / `_target`) are **ADR 0001 round 2**: building size in tokens with real polygons.

Needs `GH_TOKEN` in Colab Secrets (to install the package). CPU runtime is enough.

In [ ]:
NOTEBOOK_NAME = "p01_building_table"

# Drive layout (FYP Google account). Keep in sync with src/damagesar_bench/drive.py
import os, datetime, re
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/MyDrive/DamageSAR-Bench"
FOLDERS = ["00_admin", "01_data/raw/bright", "01_data/raw/instance_labels", "01_data/external",
           "01_data/derived", "02_models/checkpoints", "02_models/features", "03_notebooks/pipeline",
           "03_notebooks/checks", "03_notebooks/scratch", "04_outputs/runs", "04_outputs/reports"]
for f in FOLDERS:
    os.makedirs(f"{PROJECT}/{f}", exist_ok=True)
def run_dir(name):
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    p = f"{PROJECT}/04_outputs/runs/{datetime.date.today().isoformat()}_{slug}"
    os.makedirs(p, exist_ok=True)
    return p
RUN = run_dir(NOTEBOOK_NAME)
print("Drive layout ready under", PROJECT)
print("This run saves to:", RUN)

In [ ]:
# Install this repo's package (private repo: needs GH_TOKEN in Colab Secrets, read access to damagesar-bench)
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
!git clone -q https://{GH}@github.com/ThinukaLehan/damagesar-bench.git /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
!pip install -q -e /content/damagesar-bench
COMMIT = !git -C /content/damagesar-bench rev-parse --short HEAD
COMMIT = COMMIT[0]
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
import damagesar_bench
print("damagesar-bench at commit", COMMIT, "->", damagesar_bench.__file__)

In [ ]:
# Download BRIGHT SAR + labels from Hugging Face straight to /content (~3.4 GB, ~1 min).
# Copying from Drive fails (Errno 107), so always download fresh.
import os, zipfile
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # BRIGHT is not gated; the token only avoids rate limits

ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if os.path.isdir(folder) and len(os.listdir(folder)) > 4000:
        print("already there:", folder); continue
    p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
    print("unzipped", name)
for sub in ["post-event", "target"]:
    print(sub, len(os.listdir(os.path.join(ROOT, sub))), "files")

In [ ]:
import os, pandas as pd
from damagesar_bench.config import load_events, load_bench
from damagesar_bench.data.buildings import build_building_table, summarise, write_table
LABELS = f"{PROJECT}/01_data/raw/instance_labels/cvprw26_trainval_instance_labels.zip"
assert os.path.exists(LABELS), "Run c02 first: it caches the instance-label zip in Drive"
events = load_events("/content/damagesar-bench/configs/events.yaml")
cfg = load_bench("/content/damagesar-bench/configs/bench.yaml")
df, stats = build_building_table(LABELS, events, cfg.nodata, bright_root=ROOT, target_res_m=0.3)
print(stats)
summary = summarise(df)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
display(summary)

In [ ]:
# Sanity checks before saving (stop if anything looks wrong)
dev = df[df.role.isin(["development", "secondary"])]
conf = df[df.role == "confirmatory"]
assert conf.damage_class.isna().all(), "confirmatory labels must not be stored before the freeze"
assert dev.damage_class.notna().all()
print("area flags (pixel count vs polygon area, > 25% and > 4 px):", int(df.area_flag.sum()), "of", len(df))
print("BRIGHT 'area' equals the bounding box (w x h) for", int(df.reported_area_is_bbox.sum()), "of", len(df), "buildings")
agree = summary.dropna(subset=["raster_agreement"])[["event", "raster_agreement"]]
print("JSON class vs label-raster majority, per development/secondary event:")
print(agree.to_string(index=False))

In [ ]:
derived = f"{PROJECT}/01_data/derived"
write_table(df, f"{derived}/buildings.parquet")
summary.to_csv(f"{derived}/buildings_summary.csv", index=False)
summary.to_csv(f"{RUN}/buildings_summary.csv", index=False)
with open(f"{RUN}/build_info.txt", "w") as fh:
    fh.write(f"commit {COMMIT}\n{stats}\n")
print("saved", f"{derived}/buildings.parquet", "and summary;", len(df), "rows")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/pipeline/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))
